<a href="https://colab.research.google.com/github/johnyamarun/run-monitor/blob/main/Catherin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CATHERINE コアエンジン（Google Drive連携版）

In [ ]:
# =====================================================================
# セル1: プロジェクト「CATHERINE」 データ解析＆台本生成エンジン
# =====================================================================

!pip install -q fitparse google-generativeai

import os
import glob
import pandas as pd
import numpy as np
import requests
import math
from fitparse import FitFile
import google.generativeai as genai
from google.colab import userdata
from google.colab import drive

# --- 🎯 設定エリア ---
DRIVE_FOLDER_PATH = '/content/drive/MyDrive/CATHERINE_FIT'
ATHLETE_PROFILE = {
    "pb_marathon": "2時間40分12秒",
    "target_marathon": "2時間30分00秒",
    "vo2max": 67,
    "lt_pace": "3:29/km",
    "lt_hr": 158,
    "resting_hr": 45
}
# ----------------------

# Google Driveのマウント
drive.mount('/content/drive')
os.makedirs(DRIVE_FOLDER_PATH, exist_ok=True)

# Gemini API設定
try:
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
    model = genai.GenerativeModel('gemini-2.5-flash')
except userdata.SecretNotFoundError:
    print("❌ 'GEMINI_API_KEY' が見つかりません。")

# FITファイル解析
def parse_fit_to_df(file_path):
    fitfile = FitFile(file_path)
    records = []
    for record in fitfile.get_messages('record'):
        record_data = {f.name: f.value for f in record}
        records.append(record_data)
    return pd.DataFrame(records)

fit_files = glob.glob(os.path.join(DRIVE_FOLDER_PATH, '*.fit'))
if not fit_files:
    print(f"❌ {DRIVE_FOLDER_PATH} にFITファイルがありません。")
else:
    file_path = max(fit_files, key=os.path.getctime)
    print(f"✅ 対象ファイルを解析中: {os.path.basename(file_path)}")
    df = parse_fit_to_df(file_path)

    # データ抽出と変換
    target_columns = ['timestamp', 'distance', 'enhanced_speed', 'heart_rate', 'cadence', 'position_lat', 'position_long']
    df_filtered = df[[c for c in target_columns if c in df.columns]].copy().set_index('timestamp')

    def semicircles_to_degrees(s): return s * (180.0 / (2**31)) if pd.notna(s) else np.nan
    if 'position_lat' in df_filtered.columns:
        df_filtered['lat'] = df_filtered['position_lat'].apply(semicircles_to_degrees)
        df_filtered['lon'] = df_filtered['position_long'].apply(semicircles_to_degrees)
    else:
        df_filtered['lat'], df_filtered['lon'] = np.nan, np.nan

    # 気象データ取得
    temp, wind_speed_ms, wind_dir = "不明", "不明", "不明"
    idx = df_filtered['lat'].first_valid_index()
    if idx is not None:
        try:
            url = f"https://archive-api.open-meteo.com/v1/archive?latitude={df_filtered.loc[idx, 'lat']}&longitude={df_filtered.loc[idx, 'lon']}&start_date={idx.strftime('%Y-%m-%d')}&end_date={idx.strftime('%Y-%m-%d')}&hourly=temperature_2m,windspeed_10m,winddirection_10m&timezone=Asia%2FTokyo"
            weather = requests.get(url).json()
            t_idx = weather['hourly']['time'].index(idx.strftime('%Y-%m-%dT%H:00'))
            temp = weather['hourly']['temperature_2m'][t_idx]
            wind_speed_ms = round(weather['hourly']['windspeed_10m'][t_idx] / 3.6, 1)
            wind_dir = weather['hourly']['winddirection_10m'][t_idx]
            print(f"✅ 気象データ: 気温 {temp}℃, 風速 {wind_speed_ms}m/s")
        except:
            print("⚠️ 気象データ取得スキップ")

    # 1分ブロック生成
    df_1min = df_filtered.resample('1min').mean(numeric_only=True)
    df_1min_dist = df_filtered['distance'].resample('1min').max() - df_filtered['distance'].resample('1min').min()
    blocks = []
    prev_lat, prev_lon = None, None

    for ts, row in df_1min.iterrows():
        if pd.isna(row.get('enhanced_speed')) or row['enhanced_speed'] <= 0: continue

        speed, hr, cadence, dist, lat, lon = row['enhanced_speed'], row.get('heart_rate', np.nan), row.get('cadence', np.nan), df_1min_dist.loc[ts], row.get('lat'), row.get('lon')
        pace = 1000 / speed / 60

        heading = "不明"
        if prev_lat and pd.notna(lat):
            dLon = math.radians(lon - prev_lon)
            y, x = math.sin(dLon) * math.cos(math.radians(lat)), math.cos(math.radians(prev_lat)) * math.sin(math.radians(lat)) - math.sin(math.radians(prev_lat)) * math.cos(math.radians(lat)) * math.cos(dLon)
            brng = (math.degrees(math.atan2(y, x)) + 360) % 360
            if wind_dir != "不明":
                diff = abs(brng - wind_dir)
                heading = "強い追い風" if diff < 45 or diff > 315 else "強い向かい風" if 135 < diff < 225 else "横風"
        prev_lat, prev_lon = lat, lon

        blocks.append({
            "time": ts.strftime("%H:%M:%S"),
            "pace": f"{int(pace):02d}:{int((pace % 1) * 60):02d}",
            "hr": int(hr) if pd.notna(hr) else "N/A",
            "cadence": int(cadence * 2) if pd.notna(cadence) else "N/A",
            "distance_m": int(dist) if pd.notna(dist) else 0,
            "wind_status": heading
        })

 # プロンプト生成（全編）
    prompt = f"""
あなたはプロジェクト「CATHERINE」のメインAIであり、今回は「日本女子マラソン界のレジェンド（オリンピックメダリスト）」として実況解説を行います。
対象は自己ベスト{ATHLETE_PROFILE['pb_marathon']}を持ち、フルマラソン{ATHLETE_PROFILE['target_marathon']}を本気で狙う極めて走力の高いシリアスランナーです。

【ランナーの生理学的データ】VO2max: {ATHLETE_PROFILE['vo2max']}, 閾値ペース: {ATHLETE_PROFILE['lt_pace']}, 閾値心拍数: {ATHLETE_PROFILE['lt_hr']}bpm, 安静時心拍数: {ATHLETE_PROFILE['resting_hr']}bpm
【環境データ】気温{temp}度、風速{wind_speed_ms}m/s

【キャラクター設定（最重要）】
- あなたは大阪国際女子マラソンの解説席に座っているような、明るく親しみやすいレジェンド解説者です。一人称は「私」。
- 「いやー、このペースは凄いですね！」「あー、ここキツいところなんですよね、わかるわー！」といった、ランナーとしての共感と熱量を持った自然な会話調（ラジオやテレビ中継のフリートーク風）で話してください。
- ただし、ただ応援するだけでなく、メダリストならではの超一流の視点を持ってください。生理学的データ（LT心拍数{ATHLETE_PROFILE['lt_hr']}bpmなど）や気象データを基準にし、「VO2max 67の彼なら、向かい風でもこのLTペースは余裕で押していけますよ！」といった具体的な分析を必ず織り交ぜてください。
- 厳しい指摘も「ピッチが落ちてきましたね、ここが踏ん張りどころ！腕振っていきましょう！」と、愛のあるエールに変換してください。

【出力ルール】
- 以下の1分ごとのデータブロック全てに対し、1ブロックにつき約40秒（150〜200文字程度）の緻密かつ情熱的な実況を記述。
- フォーマットは必ず「[タイムスタンプ] 実況セリフ」とすること。

【データ】\n"""
    for b in blocks: prompt += f"- [{b['time']}] ペース: {b['pace']}/km, 心拍数: {b['hr']}bpm, ピッチ: {b['cadence']}spm, 移動距離: {b['distance_m']}m, 風の状況: {b['wind_status']}\n"

    print("CATHERINEが分析を開始します...\n" + "-"*40)
    response = model.generate_content(prompt)
    print(response.text)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 対象ファイルを解析中: GosenRun20260306173919.fit
✅ 気象データ: 気温 4.7℃, 風速 2.0m/s
CATHERINEが分析を開始します...
----------------------------------------
[08:39:00] さあ、始まりましたね！いやー、本当に素晴らしいコンディション！ランナーさん、まだウォーミングアップといったところでしょうか。ペースは6分42秒、心拍数も82bpmと、安静時心拍数45bpmから考えると、ゆっくり体を温めているのが分かりますね。VO2max 67の彼なら、このペースはもう、お散歩みたいなもんじゃないでしょうか？（笑）ここから徐々に上げていくんでしょうね！

[08:40:00] おっと、少しペースアップしてきましたね！6分23秒です。強い追い風を味方につけて、心拍数も104bpmと100を超えてきました。でもこのランナーの閾値心拍数158bpmからすれば、まだまだ全然余裕がありますよ。ピッチも安定していて、身体が軽そうに見えますね！いいですよー、頑張って！

[08:41:00] いやー、ぐんぐん上がってきますね！もう6分台前半、6分9秒ですよ。心拍数は落ち着いていますが、ピッチがわずかに上がってますね。身体がこの気温4.7度にも慣れてきて、どんどん動きやすくなってきているのが分かります。追い風も味方につけて、滑らかな走りです！良いリズム！

[08:42:00] 良いリズムで刻んでいますね！この6分10秒というペース、決して速いわけではないですけど、心拍数も104bpmと低いままで、本当に体が楽そうに見えます。ベテランランナーさんらしい、焦らない落ち着いた入り方ですね。目標の2時間30分に向けて、最高のスタートですよ！

[08:43:00] おー、さらにペースアップ！6分4秒まで来ましたよ！心拍数が逆に下がっているのは、体が完全に温まって、効率の良い走りを見つけたんじゃないでしょうか。素晴らしいですね。このVO2m

音声合成＆1本化エンジン

In [ ]:
# =====================================================================
# セル2: プロジェクト「CATHERINE」 音声合成＆タイムライン結合エンジン
# =====================================================================

!pip install -q edge-tts pydub

import re
import os
import edge_tts
from pydub import AudioSegment
from datetime import datetime
from IPython.display import Audio, display

# セル1で定義したドライブのフォルダに直接保存します
output_dir = os.path.join(DRIVE_FOLDER_PATH, "catherine_audio_blocks")
os.makedirs(output_dir, exist_ok=True)
final_audio_path = os.path.join(DRIVE_FOLDER_PATH, "CATHERINE_Final_Track.wav")
VOICE_MODEL = "ja-JP-NanamiNeural"

script_lines = response.text.strip().split('\n')
parsed_scripts = []
run_start_time = df_filtered.index[0]
run_date_str = run_start_time.strftime("%Y-%m-%d")

for line in script_lines:
    match = re.match(r'\[(\d{2}:\d{2}:\d{2})\]\s*(.*)', line)
    if match:
        time_str, text = match.group(1), match.group(2)
        block_time = datetime.strptime(f"{run_date_str} {time_str}", "%Y-%m-%d %H:%M:%S")
        offset_seconds = max(0, (block_time - run_start_time).total_seconds())
        parsed_scripts.append({"time_str": time_str, "text": text, "offset_ms": int(offset_seconds * 1000)})

print(f"✅ {len(parsed_scripts)}個のセリフを抽出。音声を生成します...\n" + "-"*40)

async def generate_and_assemble():
    if not parsed_scripts: return
    total_duration_ms = parsed_scripts[-1]["offset_ms"] + 60000
    base_track = AudioSegment.silent(duration=total_duration_ms)

    for i, data in enumerate(parsed_scripts):
        temp_file = os.path.join(output_dir, f"temp_{i:02d}_{data['time_str'].replace(':', '')}.mp3")
        communicate = edge_tts.Communicate(data["text"], VOICE_MODEL, rate="+10%", pitch="-5Hz")
        await communicate.save(temp_file)
        print(f"[{data['time_str']}] 音声配置 (開始: {data['offset_ms']/1000:.1f}秒)")

        clip = AudioSegment.from_mp3(temp_file)
        base_track = base_track.overlay(clip, position=data["offset_ms"])

    print("\n🎬 タイムライン結合中...")
    base_track.export(final_audio_path, format="wav")
    print(f"🎉 完了！ Google Driveに保存されました: {final_audio_path}")

await generate_and_assemble()
display(Audio(final_audio_path))

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ 68個のセリフを抽出。音声を生成します...
----------------------------------------
[08:39:00] 音声配置 (開始: 0.0秒)
[08:40:00] 音声配置 (開始: 41.0秒)
[08:41:00] 音声配置 (開始: 101.0秒)
[08:42:00] 音声配置 (開始: 161.0秒)
[08:43:00] 音声配置 (開始: 221.0秒)
[08:44:00] 音声配置 (開始: 281.0秒)
[08:45:00] 音声配置 (開始: 341.0秒)
[08:46:00] 音声配置 (開始: 401.0秒)
[08:47:00] 音声配置 (開始: 461.0秒)
[08:48:00] 音声配置 (開始: 521.0秒)
[08:49:00] 音声配置 (開始: 581.0秒)
[08:50:00] 音声配置 (開始: 641.0秒)
[08:51:00] 音声配置 (開始: 701.0秒)
[08:52:00] 音声配置 (開始: 761.0秒)
[08:53:00] 音声配置 (開始: 821.0秒)
[08:54:00] 音声配置 (開始: 881.0秒)
[08:55:00] 音声配置 (開始: 941.0秒)
[08:56:00] 音声配置 (開始: 1001.0秒)
[08:57:00] 音声配置 (開始: 1061.0秒)
[08:58:00] 音声配置 (開始: 1121.0秒)
[08:59:00] 音声配置 (開始: 1181.0秒)
[09:00:00] 音声配置 (開始: 1241.0秒)
[09:01:00] 音声配置 (開始: 1301.0秒)
[09:02:00] 音声配置 (開始: 1361.0秒)
[09:03:00] 音声配置 (開始: 1421.0秒)
[09:04:00] 音声配置 (開始: 1481.0秒)
[09:05:00] 音声配置 (開始: 1541.0秒)
[09:06:00] 音声配置 (開始: 1601.0秒)
[09:07:00] 音声配置 (開始: 1661.0秒)
[09:08:00] 音声配置 (開始: 1721.0秒)
[09:09:00] 音声配置 (開始: 1781.0秒)
[09:10:00] 音声配置 (開始: 184